앞서 한 결과에서 부터 시작해보겠습니다.

In [124]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

y = df['Survived']
X = df.drop(columns=['Survived'])

X = X.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
X['Embarked'] = X['Embarked'].map({'S': 1, 'C': 2, 'Q': 3})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 결측치를 처리할 때 사용했던 결측치 처리를 다시 나눠서 수행합니다.
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])

X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())
X_test['Embarked'] = X_test['Embarked'].fillna(X_train['Embarked'].mode()[0])

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

model.score(X_test, y_test)

c:\Users\rando\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8044692737430168

자, 모델의 정확도가 80%가 나왔죠? 모델의 정확도를 개선하기 위해선 뭐부터 하는게 좋을까요?

모델이 어느 부분에서 틀리고 어느 부분에서 오류를 범하고 있는지를 확인하는게 먼저겠죠?

In [125]:
pred = model.predict(X_test)

result = X_test.copy()
result['label'] = y_test
result['pred'] = pred

result.head(10)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,label,pred
565,3,0,24.0,2,0,24.1500,1.0,0,0
160,3,0,44.0,0,1,16.1000,1.0,0,0
553,3,0,22.0,0,0,7.2250,2.0,1,0
860,3,0,41.0,2,0,14.1083,1.0,0,0
241,3,1,28.5,1,0,15.5000,3.0,1,1
559,3,1,36.0,1,0,17.4000,1.0,1,0
387,2,1,36.0,0,0,13.0000,1.0,1,1
536,1,0,45.0,0,0,26.5500,1.0,0,0
698,1,0,49.0,1,1,110.8833,2.0,0,0
99,2,0,34.0,1,0,26.0000,1.0,0,0


여기서 모델이 틀린 데이터들을 한번 살펴보겠습니다.

In [126]:
result[result['label']!=result['pred']].head(10)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,label,pred
553,3,0,22.0,0,0,7.2250,2.0,1,0
559,3,1,36.0,1,0,17.4000,1.0,1,0
279,3,1,35.0,1,1,20.2500,1.0,1,0
712,1,0,48.0,1,0,52.0000,1.0,1,0
455,3,0,29.0,0,0,7.8958,2.0,1,0
65,3,0,28.5,1,1,15.2458,2.0,1,0
489,3,0,9.0,1,1,15.9000,1.0,1,0
869,3,0,4.0,1,1,11.1333,1.0,1,0
165,3,0,9.0,0,2,20.5250,1.0,1,0
297,1,1,2.0,1,2,151.5500,1.0,0,1


이렇게 봐서는 잘 모르겠죠? 한번 그룹 별로 나눠서 살펴 보도록 하겠습니다.

우선 남성 승객들 중에서는 어떤 승객을 잘 맞췄고 어떤 승객을 많이 틀렸는지 살펴 보도록 하겠습니다.

In [127]:
male_result = result[result['Sex']==0] # 0이 남자
pd.crosstab(male_result['label'], male_result['pred'])

pred,0,1
label,,
0,92,2
1,21,3


남성 사망자 94명 중 92명을 사망자로 예측했고, 남성 생존자 24명 중 3명만 생존했다고 예측했습니다.

In [128]:
female_result = result[result['Sex']==1] # 0이 남자
pd.crosstab(female_result['label'], female_result['pred'])

pred,0,1
label,,
0,6,10
1,2,43


여성 사망자 16명 중 6명을 사망으로 예측했고, 여성 생존자 45명 중 43명을 생존했다고 예측했습니다.

이렇게 두 결과를 보면 결과가 명백히 갈립니다. 여성은 대부분 생존으로, 남성은 대부분 사망으로 예측했다는 것을 볼 수 있습니다.

한 번 훈련셋에서 실제 성별 별 생존율을 확인해 볼까요?

In [129]:
train = X_train.copy()
train['Survived'] = y_train

train.groupby('Sex')['Survived'].mean()

Sex
0    0.185185
1    0.743083
Name: Survived, dtype: float64

보면 실제로 남성은 생존율이 20%가 안되고, 여성은 생존율이 70%가 넘어갑니다. 이렇게 학습 데이터의 통계를 반영한 어떻게 보면 당연한 결과라고 볼 수 있겠네요.

하지만 이 결과, 어떻게 개선할 수는 없을까요?

---
### Error Analysis — "어디서 틀리는가"

방금 남성 승객을 객실 등급으로 나눠 봤습니다. 그런데 이렇게 한 그룹씩 눈으로 확인하는 방식은 금방 한계에 부딪힙니다.

**틀린 행을 하나씩 들여다보는 것으로는 원인이 보이지 않습니다.** 오류를 **그룹 단위로 집계**해야 패턴이 드러납니다.

순서대로 세 가지를 봅니다.
1. 어떤 **종류**의 오류인가 (FN인가 FP인가)
2. 어떤 **그룹**에서 틀리는가 (성별 × 객실등급)
3. 모델이 **얼마나 확신할 때** 틀리는가

In [162]:
# 오류를 '종류'로 나눠 봅니다.
result['맞춤'] = (result['label'] == result['pred'])

result['오류유형'] = '정답'
result.loc[(result['label'] == 1) & (result['pred'] == 0), '오류유형'] = 'FN(생존인데 사망 예측)'
result.loc[(result['label'] == 0) & (result['pred'] == 1), '오류유형'] = 'FP(사망인데 생존 예측)'

result['오류유형'].value_counts()

오류유형
정답                144
FN(생존인데 사망 예측)     23
FP(사망인데 생존 예측)     12
Name: count, dtype: int64

전체 오류 35건 중 **FN이 23건, FP가 12건**입니다. 오류가 한쪽으로 치우쳐 있습니다.

모델이 **생존자를 놓치는 쪽(FN)으로 두 배 가까이 더 틀린다**는 뜻입니다.

> 💬 만약 이 모델이 구조 우선순위를 정하는 데 쓰인다면, FN과 FP 중 어느 쪽이 더 치명적일까요?
> 정확도 80%라는 숫자 하나로는 이 질문에 답할 수 없습니다.

In [163]:
# 성별과 객실등급으로 나눠, 그룹마다 얼마나 틀리는지 한 번에 집계합니다.
result['성별'] = result['Sex'].map({0: '남', 1: '여'})
result['틀림'] = ~result['맞춤']
result['FN'] = (result['label'] == 1) & (result['pred'] == 0)
result['FP'] = (result['label'] == 0) & (result['pred'] == 1)

error_table = result.groupby(['성별', 'Pclass']).agg(
    인원=('label', 'size'),
    실제생존율=('label', 'mean'),
    예측생존율=('pred', 'mean'),
    오류수=('틀림', 'sum'),
    FN수=('FN', 'sum'),
    FP수=('FP', 'sum'),
)
error_table['오류율'] = error_table['오류수'] / error_table['인원']

error_table.round(3)

인원  실제생존율  예측생존율  오류수  FN수  FP수    오류율
성별 Pclass                                        
남  1       30  0.367  0.167   10    8    2  0.333
   2       16  0.188  0.000    3    3    0  0.188
   3       72  0.139  0.000   10   10    0  0.139
여  1       15  0.933  1.000    1    0    1  0.067
   2       18  0.944  1.000    1    0    1  0.056
   3       28  0.500  0.714   10    2    8  0.357

이 표 한 장이 앞의 crosstab 여러 개보다 많은 것을 말해 줍니다.

**전체 오류 35건 중 20건이 딱 두 칸에 몰려 있습니다.**

| 그룹 | 인원 | 실제 생존율 | 예측 생존율 | 오류율 | 특징 |
|---|---|---|---|---|---|
| **남 1등석** | 30 | 0.37 | 0.17 | **33.3%** | FN 8건 — 살아남은 사람을 놓침 |
| **여 3등석** | 28 | 0.50 | 0.71 | **35.7%** | FP 8건 — 죽은 사람을 살았다고 함 |

나머지 네 칸은 오류율이 5~19%로 낮습니다.

여기서 **'실제 생존율'과 '예측 생존율' 두 열을 나란히 비교**하는 것이 핵심입니다.
- 남 1등석: 실제로는 37%가 살았는데 모델은 17%만 살았다고 예측 → **과소 예측**
- 여 3등석: 실제로는 50%만 살았는데 모델은 71%가 살았다고 예측 → **과대 예측**

In [164]:
# 훈련셋의 실제 생존율을 성별 x 객실등급으로 확인합니다.
train.pivot_table(index='Sex', columns='Pclass', values='Survived').round(3)

Pclass,1,2,3
Sex,,,
0,0.370,0.152,0.135
1,0.975,0.914,0.500


남성은 1등석일 때 생존율이 0.370으로 다른 객실에 비해 생존율이 2~3배 뛰고, 여성은 3등석일 때 다른 객실들에 비해 생존율이 0.5로 반토막이 납니다.

즉, 성별과 객실등급을 결합하면 새로운 정보가 발생할 수 있다는 것이죠.

그래서 해당 정보를 결합한 컬럼을 새로 만들어 보겠습니다.

In [165]:
# Sex와 Pclass를 결합한 범주형 컬럼을 만듭니다.
# 문자열로 묶어서 6개(성별 2 x 등급 3) 조합을 각각 독립적인 카테고리로 취급하게 합니다.
X2 = X.copy()
X2['Sex_Pclass'] = X2['Sex'].astype(str) + '_' + X2['Pclass'].astype(str)

X2_train, X2_test, y_train2, y_test2 = train_test_split(X2, y, test_size=0.2, stratify=y, random_state=42)

X2_train['Age'] = X2_train['Age'].fillna(X2_train['Age'].median())
X2_train['Embarked'] = X2_train['Embarked'].fillna(X2_train['Embarked'].mode()[0])
X2_test['Age'] = X2_test['Age'].fillna(X2_train['Age'].median())
X2_test['Embarked'] = X2_test['Embarked'].fillna(X2_train['Embarked'].mode()[0])

# Sex_Pclass를 원-핫 인코딩합니다. (기존 Sex, Pclass 컬럼은 남겨둬도 되지만,
# 정보가 겹치므로 여기서는 상호작용 컬럼만 추가로 사용합니다.)
X2_train = pd.get_dummies(X2_train, columns=['Sex_Pclass'])
X2_test = pd.get_dummies(X2_test, columns=['Sex_Pclass'])
X2_test = X2_test.reindex(columns=X2_train.columns, fill_value=0)  # train/test 카테고리 불일치 방지

model2 = LogisticRegression(max_iter=1000, random_state=42)
model2.fit(X2_train, y_train2)

print('기존 모델 정확도:', model.score(X_test, y_test))
print('Sex_Pclass 추가 모델 정확도:', model2.score(X2_test, y_test2))

기존 모델 정확도: 0.8044692737430168
Sex_Pclass 추가 모델 정확도: 0.8100558659217877


In [166]:
# 앞서 만든 오류 집계 로직을 함수로 묶어서, model과 model2에 똑같이 적용해 비교합니다.
def make_error_table(X_test_, y_test_, pred_):
    r = X_test_[['Sex', 'Pclass']].copy()
    r['label'] = y_test_
    r['pred'] = pred_
    r['성별'] = r['Sex'].map({0: '남', 1: '여'})
    r['틀림'] = r['label'] != r['pred']
    r['FN'] = (r['label'] == 1) & (r['pred'] == 0)
    r['FP'] = (r['label'] == 0) & (r['pred'] == 1)

    table = r.groupby(['성별', 'Pclass']).agg(
        인원=('label', 'size'),
        실제생존율=('label', 'mean'),
        예측생존율=('pred', 'mean'),
        오류수=('틀림', 'sum'),
        FN수=('FN', 'sum'),
        FP수=('FP', 'sum'),
    )
    table['오류율'] = table['오류수'] / table['인원']
    return table.round(3)

pred1 = model.predict(X_test)
pred2 = model2.predict(X2_test)

error_table_1 = make_error_table(X_test, y_test, pred1)
error_table_2 = make_error_table(X2_test, y_test2, pred2)

print('=== 기존 모델(model) 오류표 ===')
display(error_table_1)

print('=== Sex_Pclass 추가 모델(model2) 오류표 ===')
display(error_table_2)

print('=== 오류율 변화 (model2 - model) ===')
display((error_table_2['오류율'] - error_table_1['오류율']).round(3))

=== 기존 모델(model) 오류표 ===


인원  실제생존율  예측생존율  오류수  FN수  FP수    오류율
성별 Pclass                                        
남  1       30  0.367  0.167   10    8    2  0.333
   2       16  0.188  0.000    3    3    0  0.188
   3       72  0.139  0.000   10   10    0  0.139
여  1       15  0.933  1.000    1    0    1  0.067
   2       18  0.944  1.000    1    0    1  0.056
   3       28  0.500  0.714   10    2    8  0.357

=== Sex_Pclass 추가 모델(model2) 오류표 ===


인원  실제생존율  예측생존율  오류수  FN수  FP수    오류율
성별 Pclass                                        
남  1       30  0.367  0.167   10    8    2  0.333
   2       16  0.188  0.000    3    3    0  0.188
   3       72  0.139  0.000   10   10    0  0.139
여  1       15  0.933  1.000    1    0    1  0.067
   2       18  0.944  1.000    1    0    1  0.056
   3       28  0.500  0.393    9    6    3  0.321

=== 오류율 변화 (model2 - model) ===


성별  Pclass
남   1         0.000
    2         0.000
    3         0.000
여   1         0.000
    2         0.000
    3        -0.036
Name: 오류율, dtype: float64

Sex_Pclass를 추가한 결과, 여 3등급의 오류율은 35.7% → 32.1%로 줄었지만, 남 1등급은 33.3%로 변화가 없었습니다.

절반만 통했다고 해서 이 실습이 실패한 건 아닙니다. 우리는 무작정 새 컬럼을 만든 게 아니라, 오류를 그룹별로 집계해서 어디서 얼마나 틀리는지 먼저 확인했고, "성별과 등급을 결합하면 새로운 정보가 생길 것"이라는 가설을 세우고 실제로 검증했습니다. 그 결과 한쪽(여3)에서는 가설이 맞아떨어졌고, 다른 쪽(남1)에서는 통하지 않는다는 것도 함께 확인된 셈입니다.

하나 더 해볼까요?

이번엔 성별과 연령을 묶어서 살펴 보도록 하겠습니다.

In [167]:
# 나이를 구간으로 나눕니다. ('여성과 아이 먼저' 원칙을 확인하려면 '아이' 구간이 특히 중요합니다.)
age_bins = [0, 12, 18, 30, 50, 100]
age_labels = ['0-12(아동)', '13-18(청소년)', '19-30(청년)', '31-50(중장년)', '51+(노년)']

train['AgeGroup'] = pd.cut(train['Age'], bins=age_bins, labels=age_labels)

# 성별 x 연령대로 훈련셋의 실제 생존율과 인원수를 함께 확인합니다.
age_survival = train.pivot_table(index='Sex', columns='AgeGroup', values='Survived', observed=False)

print('=== 성별 x 연령대 생존율 ===')
display(age_survival.round(3))

=== 성별 x 연령대 생존율 ===


AgeGroup,0-12(아동),13-18(청소년),19-30(청년),31-50(중장년),51+(노년)
Sex,,,,,
0,0.571,0.111,0.133,0.240,0.105
1,0.630,0.731,0.739,0.766,0.917


보면 남성의 경우 아동 구간에서 생존율이 급격하게 올라가는 것을 볼 수 있습니다.

그럼 모델에서도 이런 경향이 그대로 나타날까요? 한 번 모델의 예측 결과를 살펴봅시다.

In [168]:
# 테스트셋에도 같은 연령 구간을 만들고, model2의 예측을 붙여서 남성 아동 구간만 살펴봅니다.
result2 = X2_test.copy()
result2['label'] = y_test2
result2['pred'] = pred2
result2['AgeGroup'] = pd.cut(result2['Age'], bins=age_bins, labels=age_labels)

male_child = result2[(result2['Sex'] == 0) & (result2['AgeGroup'] == '0-12(아동)')]

print('=== 남성 아동(0-12) 실제 vs 예측 ===')
display(pd.crosstab(male_child['label'], male_child['pred']))

print('인원수:', len(male_child))
print('실제 생존율:', male_child['label'].mean().round(3))
print('예측 생존율:', male_child['pred'].mean().round(3))

=== 남성 아동(0-12) 실제 vs 예측 ===


pred,0,1
label,,
0,4,0
1,4,1


인원수: 9
실제 생존율: 0.556
예측 생존율: 0.111


보면 모델의 예측 결과 생존율이 크게 떨어지는 것을 볼 수 있습니다. 모델에 해당 정보가 충분히 반영되지 않은 거죠.

연령별 그룹 컬럼을 추가해서 다시 실험해 보도록 하겠습니다.

In [169]:
# Sex_Pclass에 AgeGroup까지 결합한 컬럼을 추가로 만듭니다.
X3 = X2.copy()
X3['AgeGroup'] = pd.cut(X3['Age'], bins=age_bins, labels=age_labels)
X3['Sex_AgeGroup'] = X3['Sex'].astype(str) + '_' + X3['AgeGroup'].astype(str)
X3 = X3.drop(columns=['AgeGroup'])  # 문자열 결합에 썼으니 원본 범주형 컬럼은 제거합니다.

X3_train, X3_test, y_train3, y_test3 = train_test_split(X3, y, test_size=0.2, stratify=y, random_state=42)

X3_train['Age'] = X3_train['Age'].fillna(X3_train['Age'].median())
X3_train['Embarked'] = X3_train['Embarked'].fillna(X3_train['Embarked'].mode()[0])
X3_test['Age'] = X3_test['Age'].fillna(X3_train['Age'].median())
X3_test['Embarked'] = X3_test['Embarked'].fillna(X3_train['Embarked'].mode()[0])

# Sex_Pclass, Sex_AgeGroup을 원-핫 인코딩합니다.
# dtype=int로 지정합니다. (bool 컬럼이 섞이면 이후 shap 계산에서 타입 오류가 납니다.)
X3_train = pd.get_dummies(X3_train, columns=['Sex_Pclass', 'Sex_AgeGroup'], dtype=int)
X3_test = pd.get_dummies(X3_test, columns=['Sex_Pclass', 'Sex_AgeGroup'], dtype=int)
X3_test = X3_test.reindex(columns=X3_train.columns, fill_value=0)  # train/test 카테고리 불일치 방지

model3 = LogisticRegression(max_iter=1000, random_state=42)
model3.fit(X3_train, y_train3)

pred3 = model3.predict(X3_test)

print('기존 모델 정확도:', model.score(X_test, y_test))
print('Sex_Pclass 추가 모델 정확도:', model2.score(X2_test, y_test2))
print('Sex_Pclass+AgeGroup 추가 모델 정확도:', model3.score(X3_test, y_test3))

기존 모델 정확도: 0.8044692737430168
Sex_Pclass 추가 모델 정확도: 0.8100558659217877
Sex_Pclass+AgeGroup 추가 모델 정확도: 0.8324022346368715


In [170]:
# 테스트셋에도 같은 연령 구간을 만들고, model3의 예측을 붙여서 남성 아동 구간만 살펴봅니다.
result3 = X3_test.copy()
result3['label'] = y_test3
result3['pred'] = pred3
result3['AgeGroup'] = pd.cut(result3['Age'], bins=age_bins, labels=age_labels)

male_child = result3[(result3['Sex'] == 0) & (result3['AgeGroup'] == '0-12(아동)')]

print('=== 남성 아동(0-12) 실제 vs 예측 ===')
display(pd.crosstab(male_child['label'], male_child['pred']))

print('인원수:', len(male_child))
print('실제 생존율:', male_child['label'].mean().round(3))
print('예측 생존율:', male_child['pred'].mean().round(3))

=== 남성 아동(0-12) 실제 vs 예측 ===


pred,0,1
label,,
0,3,1
1,0,5


인원수: 9
실제 생존율: 0.556
예측 생존율: 0.667


결과를 보면 정확도도 그렇고, 남성 아동에 대한 예측율도 생존율이 크게 올라간 것을 볼 수 있습니다. 이런 식으로 여러 컬럼 사이의 관계를 살펴보거나 새로운 컬럼을 추가하는 것으로 모델의 성능에 영향을 줄 수 있습니다.